# Feature Extraction for Book Recommendations

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
books = pd.read_csv("books_preprocessed.csv")
books = books.dropna(subset=["processed_description"])
print(f"Total books: {len(books)}")
books[["title", "processed_description"]].head()

## 1. Bag of Words (BoW)

In [ ]:
count_vectorizer = CountVectorizer(max_features=5000)
bow_matrix = count_vectorizer.fit_transform(books["processed_description"])

print(f"BoW Matrix Shape: {bow_matrix.shape}")
print(f"Vocabulary Size: {len(count_vectorizer.vocabulary_)}")
print(f"Sample features: {count_vectorizer.get_feature_names_out()[:20]}")

bow_df = pd.DataFrame(bow_matrix.toarray()[:5, :10], 
                       columns=count_vectorizer.get_feature_names_out()[:10],
                       index=books["title"].iloc[:5])
print("\nSample BoW vectors (first 5 books, first 10 features):")
bow_df

## 2. TF-IDF Vectorization

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(books["processed_description"])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Vocabulary Size: {len(tfidf_vectorizer.vocabulary_)}")

tfidf_df = pd.DataFrame(tfidf_matrix.toarray()[:5, :10],
                         columns=tfidf_vectorizer.get_feature_names_out()[:10],
                         index=books["title"].iloc[:5])
print("\nSample TF-IDF vectors (first 5 books, first 10 features):")
tfidf_df

In [ ]:
tfidf_cosine_sim = cosine_similarity(tfidf_matrix)
print(f"Cosine Similarity Matrix Shape: {tfidf_cosine_sim.shape}")

def tfidf_recommend(title, top_k=5):
    if title not in books["title"].values:
        return "Book not found"
    idx = books[books["title"] == title].index[0]
    sim_scores = list(enumerate(tfidf_cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_k+1]
    book_indices = [i[0] for i in sim_scores]
    results = books.iloc[book_indices][["title", "authors"]].copy()
    results["similarity_score"] = [sim_scores[i][1] for i in range(len(sim_scores))]
    return results

print("\nTF-IDF Recommendations for first book:")
tfidf_recommend(books["title"].iloc[0])

## 3. Sentence Transformer Embeddings (all-MiniLM-L6-v2)

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

descriptions = books["description"].fillna("").tolist()
print(f"\nEncoding {len(descriptions)} book descriptions...")
embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=32)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
def semantic_recommend(query, top_k=5):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    results = books.iloc[top_indices][["title", "authors"]].copy()
    results["similarity_score"] = similarities[top_indices]
    return results

print("Semantic search: 'dark fantasy with dragons'")
print(semantic_recommend("dark fantasy with dragons"))
print("\nSemantic search: 'romantic story set in Europe'")
print(semantic_recommend("romantic story set in Europe"))

## 4. Save Artifacts for Dashboard

In [ ]:
np.save("embeddings.npy", embeddings)

with open("tfidf_artifacts.pkl", "wb") as f:
    pickle.dump({
        "vectorizer": tfidf_vectorizer,
        "matrix": tfidf_matrix
    }, f)

books.to_csv("books_preprocessed.csv", index=False)

print("Saved:")
print("  - embeddings.npy (Sentence Transformer embeddings)")
print("  - tfidf_artifacts.pkl (TF-IDF vectorizer + matrix)")
print("  - books_preprocessed.csv (preprocessed book data)")